# ControlNet — Adding Conditional Control to Text-to-Image Diffusion Models (2023)

_Papers · Generative Models_

**Steer a frozen diffusion model with edges, depth, or pose by attaching a trainable copy through zero-initialized convolutions.**

---

This notebook is a practice scaffold for the **ControlNet — Adding Conditional Control to Text-to-Image Diffusion Models (2023)** lesson. We build the idea one piece at a time: a zero convolution that outputs nothing yet still learns, a frozen backbone wrapped by a trainable copy, and a short training run where the control strength grows in from zero. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

### Step 1 — A zero convolution: zero output, nonzero gradient

A *zero convolution* is a 1x1 convolution whose weight **and** bias start at zero, so it outputs exactly `0` no matter the input. The key subtlety is that its weight gradient is **not** zero: the gradient with respect to each weight is the layer's input, which is generally nonzero. That is why the layer outputs nothing at init yet can still learn to pass a signal — the building block of Eq. 3.

In [ ]:
import torch
import torch.nn as nn
import copy

torch.manual_seed(0)

# A 1x1 convolution, 3 channels -> 3 channels.
zc = nn.Conv2d(3, 3, 1)

# ZERO the weight AND the bias so the layer outputs exactly 0 at init.
nn.init.zeros_(zc.weight)
nn.init.zeros_(zc.bias)

x_demo = torch.randn(1, 3, 4, 4)

# The output is 0 everywhere (Eq. 3 building block).
zc_output = zc(x_demo)
print("zero-conv output max|.|      :", zc_output.abs().max().item())   # -> 0.0

# But the WEIGHT gradient is nonzero, so the layer WILL learn.
out = zc(x_demo)
out.sum().backward()
print("zero-conv weight.grad max|.| :", round(zc.weight.grad.abs().max().item(), 2))  # -> ~6.5

### Step 2 — Build the ControlNet block (Eq. 2)

ControlNet keeps a **frozen** copy of a pretrained block (the locked backbone, `F(.;Theta)`) and a **trainable** deep copy of it (`F(.;Theta_c)`). Two zero convolutions bracket the trainable copy: `z1` gates the conditioning signal going in, `z2` gates its contribution coming out. The forward pass implements Eq. 2: `y_c = F(x;Theta) + Z( F(x + Z(c;Tz1); Theta_c); Tz2)`.

In [ ]:
# A tiny stand-in for a pretrained diffusion block (the LOCKED COPY).
def block():
    return nn.Sequential(
        nn.Conv2d(3, 8, 3, padding=1),
        nn.ReLU(),
        nn.Conv2d(8, 3, 3, padding=1),
    )

class ControlNet(nn.Module):
    def __init__(self, frozen, zero_init=True):
        super().__init__()
        # F(.;Theta) -- the locked copy; its parameters never update.
        self.frozen = frozen
        for p in self.frozen.parameters():
            p.requires_grad_(False)

        # F(.;Theta_c) -- a trainable deep copy of the same block.
        self.copy = copy.deepcopy(frozen)
        for p in self.copy.parameters():
            p.requires_grad_(True)

        # Two zero convolutions: one on the condition path, one on the output path.
        self.z1 = nn.Conv2d(3, 3, 1)
        self.z2 = nn.Conv2d(3, 3, 1)
        if zero_init:
            for z in (self.z1, self.z2):
                nn.init.zeros_(z.weight)
                nn.init.zeros_(z.bias)

    def forward(self, x, c):
        # Eq. 2:  y_c = F(x;Theta) + Z( F(x + Z(c;Tz1); Theta_c); Tz2)
        conditioned_input = x + self.z1(c)
        control_branch = self.z2(self.copy(conditioned_input))
        return self.frozen(x) + control_branch

frozen = block()
cn = ControlNet(frozen, zero_init=True)

x = torch.randn(4, 3, 8, 8)   # the input feature map
c = torch.randn(4, 3, 8, 8)   # the conditioning signal (edges, depth, pose, ...)

### Step 3 — Verify the no-op start, then train the control in (Eq. 3)

At initialization the zero convolutions make the control branch output zero, so the conditioned output **equals** the frozen output — adding the branch does no harm (Eq. 3). We then train the control branch to follow a toy target (the frozen output plus the conditioning signal) and watch the **control strength** — how far the output moves off the frozen output — grow smoothly from zero.

In [ ]:
# Eq. 3: at init the conditioned output EQUALS the frozen output.
y  = cn.frozen(x)
yc = cn(x, c)
print("max|y_c - y| at init        :", (yc - y).abs().max().item())          # -> 0.0
print("allclose(y_c, frozen(x))    :", torch.allclose(yc, y, atol=1e-6))     # -> True

# Toy target: the frozen output PLUS the control signal.
target = (cn.frozen(x) + c).detach()

def control_strength():
    # How far the output has moved off the frozen output (RMS).
    with torch.no_grad():
        return (cn(x, c) - cn.frozen(x)).pow(2).mean().sqrt().item()

trainable_params = [p for p in cn.parameters() if p.requires_grad]
opt = torch.optim.Adam(trainable_params, lr=1e-2)
lossfn = nn.MSELoss()

print("\nstep | control strength | loss")
for step in range(401):
    if step in (0, 1, 5, 20, 100, 400):
        loss_now = lossfn(cn(x, c), target).item()
        print("%4d |      %.4f      | %.4f" % (step, control_strength(), loss_now))
    opt.zero_grad()
    loss = lossfn(cn(x, c), target)
    loss.backward()
    opt.step()
# step | control strength | loss
#    0 |      0.0000      | 1.0092   <- zero effect at init (Eq. 3)
#    1 |      0.0095      | 1.0079
#    5 |      0.0552      | 0.9937
#   20 |      0.5127      | 0.7376
#  100 |      0.9099      | 0.1890
#  400 |      0.9687      | 0.0745   <- conditioning effect has grown in
# (Our small run, not the paper's reported numbers. Values vary by seed/hardware.)

### Step 4 — Ablation: random-init zero convolutions break the safe start

What if the two zero convolutions were initialized randomly instead of at zero? Then they output nonzero values at init, the control branch perturbs the frozen output before learning anything, and Eq. 3 no longer holds. This injects unstructured noise into the pretrained features on the very first step — exactly the "harmful noise" the zero initialization eliminates.

In [ ]:
# Random-init zero-convs break Eq. 3 and perturb the frozen output at step 0.
cn_rand = ControlNet(block(), zero_init=False)
with torch.no_grad():
    harm = (cn_rand(x, c) - cn_rand.frozen(x)).abs().max().item()
print("random-init max|y_c - y| at init:", round(harm, 4))   # -> ~1.13  (harmful noise at step 0)
print("zero-init  max|y_c - y| at init: 0.0   (safe start)")

## Visualize the data & results

_As we train the control branch, how does the conditioning effect grow from zero — and what does random-init do instead at step zero?_

### Step 1 — Rebuild the block and a training run that records the curve

This panel re-runs the same ControlNet on its own seed so the figure is self-contained. We rebuild the block and the `ControlNet` module, then train for 401 steps, recording the **control strength** at a few checkpoints so we can show how the conditioning effect grows in from zero.

In [ ]:
import torch
import torch.nn as nn
import copy

torch.manual_seed(0)

def block():
    return nn.Sequential(
        nn.Conv2d(3, 8, 3, padding=1),
        nn.ReLU(),
        nn.Conv2d(8, 3, 3, padding=1),
    )

class ControlNet(nn.Module):
    def __init__(self, frozen, zero_init=True):
        super().__init__()
        self.frozen = frozen
        for p in self.frozen.parameters():
            p.requires_grad_(False)
        self.copy = copy.deepcopy(frozen)
        for p in self.copy.parameters():
            p.requires_grad_(True)
        self.z1 = nn.Conv2d(3, 3, 1)
        self.z2 = nn.Conv2d(3, 3, 1)
        if zero_init:
            for z in (self.z1, self.z2):
                nn.init.zeros_(z.weight)
                nn.init.zeros_(z.bias)

    def forward(self, x, c):                       # Eq. 2
        conditioned_input = x + self.z1(c)
        control_branch = self.z2(self.copy(conditioned_input))
        return self.frozen(x) + control_branch

cn = ControlNet(block(), zero_init=True)
x = torch.randn(4, 3, 8, 8)
c = torch.randn(4, 3, 8, 8)
target = (cn.frozen(x) + c).detach()

def strength():
    with torch.no_grad():
        return (cn(x, c) - cn.frozen(x)).pow(2).mean().sqrt().item()

opt = torch.optim.Adam([p for p in cn.parameters() if p.requires_grad], lr=1e-2)
lf = nn.MSELoss()
curve = {}
for step in range(401):
    if step in (0, 1, 5, 20, 100, 400):
        curve[step] = round(strength(), 4)
    opt.zero_grad()
    loss = lf(cn(x, c), target)
    loss.backward()
    opt.step()
print("control strength:", curve)
# control strength: {0: 0.0, 1: 0.0095, 5: 0.0552, 20: 0.5127, 100: 0.9099, 400: 0.9687}

### Step 2 — Compare zero-init vs random-init at step zero

Finally, contrast the two initializations at the first step. The trained zero-init model leaves the frozen output untouched at init, while a fresh random-init model injects roughly 1.13 of noise into the same frozen output — the gap that motivates ControlNet's zero convolutions.

In [ ]:
# Ablation: harm at step 0, zero-init vs random-init.
cn_r = ControlNet(block(), zero_init=False)
with torch.no_grad():
    zero_harm = (cn(x, c) - cn.frozen(x)).abs().max().item()
    rand_harm = (cn_r(x, c) - cn_r.frozen(x)).abs().max().item()
print("zero-init  max|y_c-y| @0:", round(zero_harm, 4))   # 0.0 (post-train; at init it was 0 too)
print("random-init max|y_c-y| @0:", round(rand_harm, 4))  # ~1.132
# zero-init keeps the frozen output untouched at step 0; random-init injects ~1.13 of noise.
# Our small run, not the paper's number.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The zero-init ablation. You have a working ControlNet block whose two zero convolutions start at
            zero, so $\mathbf{y}_c = \mathbf{y}$ at step zero. Now initialize the two $1\times1$ convolutions
            randomly instead (the default PyTorch init), everything else identical. What happens to the
            step-zero output, and what does that do to the pretrained backbone?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Replace nn.init.zeros_(z.weight); nn.init.zeros_(z.bias) with the default random init (skip the zeroing). — _Random weights make $\mathcal{Z}(\cdot;\cdot)$ output nonzero values at init, so the second summand of Eq. 2 is no longer zero._
- Re-check the step-zero output: compare model(x,c) to frozen(x). — _Eq. 3 no longer holds: $\mathbf{y}_c \ne \mathbf{y}$. The control branch perturbs the frozen output before it has learned anything._
- Reason about the consequence: the random control branch injects unstructured noise into the pretrained features on the first step, the very thing zero-init prevents. — _The abstract's claim &mdash; "no harmful noise could affect the finetuning" &mdash; relies on the zero-init no-op. Random init breaks that guarantee._

**Answer:** With random init the zero convolutions are no longer zero, so
                 $\mathbf{y}_c \ne \mathbf{y}$ at step zero: the control branch adds noise to the frozen
                 output before learning anything useful. In our run the random-init output differs from the frozen
                 output by a max of about 1.13 at step zero, versus exactly 0.0 for zero-init. That is
                 precisely the "harmful noise" ControlNet's zero convolutions eliminate &mdash; the safe start comes
                 from the zero initialization, not from anything else in the architecture. (Our small run, not the
                 paper's number.)

</details>

**Problem 2.** Why does freezing the original block (the locked copy) and training a separate copy protect the
            pretrained model, when a naive fine-tune on the same small conditioning dataset might damage it?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Write what naive fine-tuning does: it updates the original parameters $\Theta$ directly using gradients from the small new dataset. — _Small datasets give noisy gradients; applied straight to the billions-of-images backbone, they can overwrite useful features (catastrophic forgetting)._
- Write what ControlNet does: $\Theta$ is frozen; only the copy $\Theta_c$ and the zero convolutions are trained, and their effect is added to the frozen output (Eq. 2). — _The backbone is never touched, so it cannot be corrupted. The new pathway can only add a correction on top._
- Add the zero-init guarantee: that correction starts at zero (Eq. 3), so even the added pathway does no harm on step one. — _Freezing protects the backbone; zero-init protects the very first steps of the new pathway. Two layers of safety._

**Answer:** Because the pretrained weights $\Theta$ are frozen and the new capability lives in a
                 separate trainable copy whose contribution is added on top (Eq. 2). Naive fine-tuning
                 mutates $\Theta$ directly with noisy small-dataset gradients and can erase what the backbone knew.
                 ControlNet never updates $\Theta$, so the backbone is preserved exactly; and the added control
                 branch starts as a no-op (zero convolutions, Eq. 3), so it does no harm at step zero either. The
                 backbone is protected by freezing; the connection is protected by zero-init.

</details>

**Problem 3.** A teammate worries: "If the zero convolution's weight starts at exactly zero, gradient descent will
            leave it at zero forever, so the conditioning can never have any effect." Is this right? Show the
            gradient.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Write one output channel of the zero convolution at a pixel: $o = \sum_i w_i p_i + b$, with all $w_i=0$ and $b=0$ at init, and input channels $p_i$. — _This is just a per-pixel linear map &mdash; the $1\times1$ convolution._
- Differentiate with respect to the weight: $\partial o / \partial w_i = p_i$. — _The gradient flowing into $w_i$ is the input $p_i$, which is generally NONZERO &mdash; it does not depend on $w_i$ being zero._
- Conclude: after one step the optimizer moves $w_i$ off zero by an amount proportional to $p_i$ (times the upstream gradient), so the layer stops being a no-op. — _The "stuck at zero" intuition confuses the gradient with respect to the input ($=w_i=0$ at init) with the gradient with respect to the weight ($=p_i\ne0$)._

**Answer:** No &mdash; the worry confuses two gradients. The gradient with respect to the weight is the
                 layer's input, $\partial o/\partial w_i = p_i$, which is nonzero. So the optimizer moves the
                 weight off zero on the very first step and the zero convolution begins to pass a signal &mdash; the
                 parameters "progressively grow from zero." (What is zero at init is the gradient with respect
                 to the input, $w_i=0$, which is exactly why no noise leaks into the trainable copy on step one.) In
                 our run the zero convolution's output is exactly $0$ at init while its weight gradient has magnitude
                 about 6.5 &mdash; nonzero, so it trains. (Our small run, not the paper's number.)

</details>